In [12]:
#!/usr/bin/env python3
"""
Chapter1_Data_Preparation_and_Spatial_Pipeline.py (cleaned & hardened)

Purpose:
  - Prepare oil & gas assets CSV (template if absent)
  - Snap assets to pipeline/transmission geometries (local files preferred; OSM fallback)
  - Compute pipeline lengths, build NetworkX graph and centralities
  - Join Zenodo incident data to compute events-within-10km and nearest distances
  - Output GeoPackage / GeoJSON / CSV artifacts for Chapter 1 modeling

Notes:
  - This version adds robust autodiscovery of pipeline / transmission files under data/raw/
    (supports .zip containing GeoJSON, plain .geojson, shapefile sets, and single .shp if its companions are present).
  - Improved OSMnx fallbacks: tries geometries_from_polygon / geometries_from_bbox compatibility.
  - Safer handling when inputs are missing; script will produce partial outputs rather than crash.
"""

import csv
import json
import logging
import re
import sys
import traceback
from pathlib import Path
from typing import Optional, Tuple, List

import geopandas as gpd
import networkx as nx
import numpy as np
import pandas as pd
from shapely.geometry import LineString, Point
import shapely.speedups

shapely.speedups.enable()

# logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
log = logging.getLogger("chapter1_pipeline")

# -----------------------------
# CONFIG
# -----------------------------
DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
PROC_DIR = DATA_DIR / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)

# Zenodo events CSV path (place the file here)
ZENODO_CSV_PATH = RAW_DIR / "combined_energy_attacks_02.csv"

# Optional explicit file paths you can set (leave None to allow autodiscovery)
LOCAL_PIPELINES_PATH: Optional[Path] = None  # e.g. RAW_DIR / "global_oil_and_gas_pipeline_grid.zip"
LOCAL_TRANSMISSION_PATH: Optional[Path] = None  # e.g. RAW_DIR / "nigeria_transmission.zip" or .shp

# Outputs
ASSETS_CSV = RAW_DIR / "oil_gas_assets.csv"
ASSETS_GPKG = PROC_DIR / "oil_gas_assets.gpkg"
PIPELINES_GEOJSON = PROC_DIR / "pipelines_snapped.geojson"
PIPELINES_WITH_LENGTHS = PROC_DIR / "pipelines_with_lengths.geojson"
NODES_GPKG = PROC_DIR / "nodes.gpkg"
NODES_CSV = PROC_DIR / "nodes_centralities.csv"
EXPOSURE_CSV = PROC_DIR / "exposure_sample.csv"
SUMMARY_JSON = PROC_DIR / "processing_summary.json"

# Template asset CSV (unchanged)
ASSETS_CSV_CONTENT = r"""Asset_ID,Name,Type,Latitude,Longitude,Capacity,Units,State_Region,Source_URL,Confidence,Notes
R-DNGT,Dangote Refinery (Ibeju-Lekki),Refinery/Integrated Complex,6.4590,4.0210,650000,bbl/d,Lagos,https://www.dangote.com/operations/refinery,High,"Refinery capacity widely reported; fertilizer cluster colocated"
F-DNGT,Dangote Fertiliser Plant (Ibeju-Lekki),Fertiliser/Ammonia,6.4419,4.0496,2800000,tpa,Lagos,https://fertiliser.dangote.com/,High,"Urea capacity reported ~2.8–3.0 Mtpa"
R-PHRC,Port Harcourt Refining Complex (PHRC),Refinery,4.7600,7.1055,210000,bbl/d,Rivers,https://globalenergyobservatory.org/,Medium,"Combined historic units; capacity approximate"
R-WAR,Warri Refinery & Petrochemicals (WRPC),Refinery,5.5680,5.7170,125000,bbl/d,Delta,https://globalenergyobservatory.org/,Medium,"Design/nominal capacity; modernisation ongoing"
R-KAD,Kaduna Refinery & Petrochemical Company (KRPC),Refinery,10.4116,7.4907,110000,bbl/d,Kaduna,https://globalenergyobservatory.org/,High,"Design capacity reported; confirm current operability"
T-BON,Bonny Island Terminal (NLNG area),Export Terminal / LNG hub,4.4428,7.2373,, ,Rivers,https://nlng.com/,High,"Major LNG & export terminal cluster (NLNG & Bonny operations)"
T-FORC,Forcados Terminal,Export Terminal,5.3667,5.4333,, ,Delta,https://ports.marinelink.com/,Medium,"Major crude export terminal area"
T-ESCR,Escravos Terminal / Escravos Gas Plant,Export Terminal / Gas Plant,5.6253,5.1924,, ,Delta,https://en.wikipedia.org/wiki/Escravos,High,"Escravos export & gas processing cluster (EGTL/Chevron)"
P-OBEN,Obiafu-Obrikom-Oben (OB3) node,Pipeline Node (gas),5.8700,5.6750,, ,Delta/Edo,https://www.gem.wiki/Obiafu-Obrikom-Oben_Gas_Pipeline,Medium,"Coordinates approximate — must snap to pipeline geometry"
P-ESCR_GP,Escravos Gas Plant (EGP / EGTL),Gas Processing Plant,5.6253,5.1924,, ,Delta,https://en.wikipedia.org/wiki/Escravos,Medium,"Same locality as Escravos terminal; capacity site-specific"
P-BON_PSP,Nembe / Nembe Creek trunk node (NCTL region),Pipeline Node (crude),4.8500,6.2500,150000,bpd,Rivers,https://www.gem.wiki/Nembe_Creek_Trunk_Line,Medium,"Approx center for NCTL; snap to line geometry"
P-TNP_HEAD,Trans-Niger Pipeline (TNP) node / Rumuekpe area,Pipeline Node (crude),,,180000,bpd,Rivers,https://www.gem.wiki/Trans_Niger_Oil_Pipeline,Medium,"Exact node coords variable; add pumping station points from pipeline shapefile"
S-PRM_PS,Major Pumping / Tank Farm (e.g., Bonny/Forcados pumping stations),Pumping Station / Tank Farm,4.4428,7.2373,, ,Rivers/Delta,https://ports.marinelink.com/,Medium,"Use terminal coords; refine when pipeline shapefile available"
PL-NCTL,Nembe Creek Trunk Line (approx center),Pipeline (oil),4.8500,6.2500,150000,bpd,Rivers,https://www.gem.wiki/Nembe_Creek_Trunk_Line,High,"Pipeline feature — line geometry required; this row is a center placeholder"
PL-ELPS,Escravos–Lagos Pipeline System (ELPS center),Pipeline (gas),5.6253,5.1924,800,MMSCFD,Delta/Lagos,https://en.wikipedia.org/wiki/Escravos%E2%80%93Lagos_Pipeline_System,High,"ELPS corridor center placeholder; snap to line to derive endpoints/length"
PL-TNP,Trans-Niger Pipeline (TNP center),Pipeline (oil),,,180000,bpd,Delta/Rivers,https://www.gem.wiki/Trans_Niger_Oil_Pipeline,Medium,"Historic TNP corridors; use operator maps/OSM for exact geometry"
PL-OB3,Obiafu-Obrikom-Oben (OB3 pipeline),Pipeline (gas),5.8700,5.6750,, ,Delta/Edo,https://www.gem.wiki/Obiafu-Obrikom-Oben_Gas_Pipeline,Medium,"Project pipeline (OB3) — geometry may be segmented"
PL-FORC_EQ,Forcados export corridor (terminal link),Pipeline (oil),5.3667,5.4333,, ,Delta,https://ports.marinelink.com/,Medium,"Forcados export corridor; snap to local segments"
PL-BONNY_EXPORT,Bonny export pipeline network,Pipeline (oil/gas),4.4428,7.2373,, ,Rivers,https://nlng.com/,High,"Bonny export network (NLNG, trunklines)"
"""

# -----------------------------
# Utilities: CSV helpers (unchanged, small improvements)
# -----------------------------
def is_csv_valid(path: Path) -> bool:
    try:
        df = pd.read_csv(path, nrows=0)
        return len(df.columns) >= 5
    except Exception:
        return False


def detect_delimiter(sample_text: str) -> str:
    """Detect delimiter using a sample (first 8 KB) then fallback heuristics."""
    try:
        sniffer = csv.Sniffer()
        sample = sample_text[:8192]
        dialect = sniffer.sniff(sample, delimiters=[",", "\t", ";", "|"])
        return dialect.delimiter
    except Exception:
        first_line = sample_text.splitlines()[0] if sample_text else ""
        if "\t" in first_line and first_line.count("\t") > first_line.count(","):
            return "\t"
        return ","


def normalize_header_names(cols):
    mapping = {}
    simple = [re.sub(r"[^a-z0-9]", "", c.lower()) for c in cols]
    for orig, s in zip(cols, simple):
        if s in ("assetid", "asset_id", "id"):
            mapping[orig] = "Asset_ID"
        elif "name" in s and "asset" not in s:
            mapping[orig] = "Name"
        elif s in ("type", "assettype"):
            mapping[orig] = "Type"
        elif re.search(r"\blat\b", s):
            mapping[orig] = "Latitude"
        elif re.search(r"\blon\b|\blong\b", s):
            mapping[orig] = "Longitude"
        elif "capacity" in s:
            mapping[orig] = "Capacity"
        elif s in ("units", "unit"):
            mapping[orig] = "Units"
        elif "state" in s or "region" in s:
            mapping[orig] = "State_Region"
        elif "source" in s:
            mapping[orig] = "Source_URL"
        elif "confidence" in s:
            mapping[orig] = "Confidence"
        elif "note" in s:
            mapping[orig] = "Notes"
        else:
            mapping[orig] = orig
    return mapping


def safe_read_csv_with_lastcol_join(path: Path, expected_fields=None, encoding="utf-8") -> pd.DataFrame:
    """
    Robust CSV read:
      - try pandas read with detected delimiter
      - on ParserError, write a cleaned file merging trailing fields into last column
      - normalize headers
    """
    text = path.read_text(encoding=encoding)
    delim = detect_delimiter(text)
    try:
        df = pd.read_csv(path, sep=delim, engine="python", dtype=str, encoding=encoding)
        mapping = normalize_header_names(list(df.columns))
        df = df.rename(columns=mapping)
        return df
    except pd.errors.ParserError:
        log.warning("ParserError with detected delimiter; attempting robust cleaning.")
    except Exception as e:
        log.warning("Read attempt failed: %s", e)

    # fallback cleaning
    lines = text.splitlines()
    if not lines:
        raise RuntimeError("CSV file is empty.")
    header_raw = lines[0].rstrip("\n")
    header_parts = header_raw.split(delim) if delim else header_raw.split(",")
    header_parts = [h.strip().strip("\ufeff") for h in header_parts]
    if expected_fields is None:
        expected_fields = len(header_parts)
    cleaned_path = Path(str(path) + ".cleaned")
    with cleaned_path.open("w", encoding=encoding, newline="") as fout:
        writer = csv.writer(fout)
        writer.writerow(header_parts)
        for raw in lines[1:]:
            parts = raw.rstrip("\n").split(delim, expected_fields - 1) if delim else raw.rstrip("\n").split(",", expected_fields - 1)
            if len(parts) < expected_fields:
                parts += [""] * (expected_fields - len(parts))
            parts = [p.strip() for p in parts]
            writer.writerow(parts)
    log.info("Cleaned CSV written to: %s", cleaned_path)
    df = pd.read_csv(cleaned_path, dtype=str, encoding=encoding)
    mapping = normalize_header_names(list(df.columns))
    df = df.rename(columns=mapping)
    return df


# -----------------------------
# Asset loader uses robust CSV read + normalization
# -----------------------------
def load_assets(csv_path: Path) -> gpd.GeoDataFrame:
    df = safe_read_csv_with_lastcol_join(csv_path)
    expected_cols = [
        "Asset_ID",
        "Name",
        "Type",
        "Latitude",
        "Longitude",
        "Capacity",
        "Units",
        "State_Region",
        "Source_URL",
        "Confidence",
        "Notes",
    ]
    detected = list(df.columns)
    missing = [c for c in expected_cols if c not in detected]
    if missing:
        log.error("Detected header columns: %s", detected)
        raise RuntimeError(
            f"Assets CSV missing expected columns: {missing}\nPlease check the header names or upload the CSV. Detected header printed above."
        )
    # preserve raw capacity
    df["Capacity_raw"] = df["Capacity"].astype(str)
    # try to extract numeric capacity where possible (strip non-digit chars)
    df["Capacity"] = pd.to_numeric(df["Capacity"].astype(str).str.replace(r"[^\d\.\-]", "", regex=True), errors="coerce")
    df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce")
    df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce")
    # create geometry (if lat/lon present)
    df["geometry"] = df.apply(
        lambda r: Point(float(r["Longitude"]), float(r["Latitude"])) if pd.notna(r["Latitude"]) and pd.notna(r["Longitude"]) else None,
        axis=1,
    )
    gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")
    return gdf


# -----------------------------
# Zenodo events loader (robust)
# -----------------------------
def load_zenodo_events(path_or_url) -> gpd.GeoDataFrame:
    p = Path(path_or_url)
    if p.exists():
        df = pd.read_csv(p, dtype=str, encoding="utf-8")
    else:
        try:
            df = pd.read_csv(str(path_or_url))
        except Exception as e:
            raise RuntimeError(f"Could not load Zenodo events file locally or from URL: {e}")
            
    # IMPROVED: Use regex to find Latitude/Longitude more strictly
    # This prevents matching "related" or "lation"
    lat_cols = [c for c in df.columns if re.search(r'\blat(itude)?\b', c.lower())]
    lon_cols = [c for c in df.columns if re.search(r'\blon(gitude)?\b', c.lower())]
    
    # Fallback: if strict search fails, try the broader search but prioritize "Latitude"
    if not lat_cols:
        lat_cols = [c for c in df.columns if "lat" in c.lower() and "related" not in c.lower()]
    if not lon_cols:
        lon_cols = [c for c in df.columns if "lon" in c.lower() or "long" in c.lower()]

    if lat_cols and lon_cols:
        # Use the first valid match found
        df["latitude"] = pd.to_numeric(df[lat_cols[0]], errors="coerce")
        df["longitude"] = pd.to_numeric(df[lon_cols[0]], errors="coerce")
    else:
        raise RuntimeError(f"Could not find coordinate columns. Detected columns: {list(df.columns)}")
    
    # Drop invalid rows
    df = df.dropna(subset=["latitude", "longitude"])
    
    if len(df) == 0:
        print("WARNING: All rows were dropped after coordinate conversion. Check data formats.")

    gdf = gpd.GeoDataFrame(
        df, 
        geometry=gpd.points_from_xy(df.longitude, df.latitude), 
        crs="EPSG:4326"
    )
    return gdf

# -----------------------------
# File autodiscovery for pipeline / transmission geometries
# -----------------------------
def autodiscover_line_files(raw_dir: Path) -> List[Path]:
    """
    Look in raw_dir for candidate files:
      - any .zip (often zipped geojson)
      - any .geojson, .json
      - any shapefile set (.shp)
      - patterns containing 'pipeline'|'pipelines'|'transmission'|'power'
    Returns list of candidate Paths ordered by preference.
    """
    candidates = []
    patterns = ["*pipeline*.zip", "*pipelines*.zip", "*transmission*.zip", "*power*.zip", "*.zip",
                "*pipeline*.geojson", "*pipelines*.geojson", "*transmission*.geojson", "*.geojson",
                "*transmission*.shp", "*pipeline*.shp", "*.shp"]
    for pat in patterns:
        for p in raw_dir.glob(pat):
            if p not in candidates:
                candidates.append(p)
    # also include any other geojson/shp if nothing found (fallback)
    if not candidates:
        for p in raw_dir.glob("*.geojson"):
            candidates.append(p)
        for p in raw_dir.glob("*.shp"):
            if p not in candidates:
                candidates.append(p)
        for p in raw_dir.glob("*.zip"):
            if p not in candidates:
                candidates.append(p)
    return candidates


def try_read_vector(path: Path) -> Optional[gpd.GeoDataFrame]:
    """
    Try to read a vector file robustly:
      - geopandas.read_file(path) often handles zipped geojson or zipped shapefiles
      - If path is .shp but sidecars missing, attempt to find same-stem files in same folder
    """
    try:
        log.info("Attempting to read vector file: %s", path)
        gdf = gpd.read_file(path)
        if gdf is None or len(gdf) == 0:
            return gdf
        if gdf.crs is None:
            try:
                gdf.set_crs(epsg=4326, inplace=True)
            except Exception:
                pass
        return gdf
    except Exception as e:
        log.debug("Failed reading %s: %s", path, e)
        return None


# -----------------------------
# OSMnx fetch with robust fallbacks
# -----------------------------
def _get_osmnx_geometries_for_nigeria(pipe_tag: dict, power_tag: dict) -> gpd.GeoDataFrame:
    """
    Robustly query OSM for pipeline & powerline geometries covering Nigeria.
    Uses polygon-based fetch (geocode Nigeria then fetch geometries within that polygon) to avoid
    versions where geometries_from_bbox is not exposed.
    """
    try:
        import osmnx as ox
    except Exception as e:
        raise RuntimeError(f"osmnx is not importable: {e}. Install with conda-forge: conda install -c conda-forge osmnx") from e

    # first get Nigeria polygon by name (geocode_to_gdf is present across many versions)
    try:
        nigeria_gdf = ox.geocode_to_gdf("Nigeria")
        if nigeria_gdf is None or nigeria_gdf.empty:
            raise RuntimeError("Could not geocode Nigeria polygon from OSM.")
        nigeria_poly = nigeria_gdf.unary_union
    except Exception as e:
        raise RuntimeError(f"Could not obtain Nigeria polygon from OSM: {e}")

    combined = []
    # attempt geometries_from_polygon (preferred), otherwise try geometries_from_bbox fallback
    for tags in (pipe_tag, power_tag):
        fetched = None
        # try direct function locations
        try:
            if hasattr(ox, "geometries_from_polygon"):
                fetched = ox.geometries_from_polygon(nigeria_poly, tags)
            else:
                # try submodule name
                try:
                    from osmnx import geometries as oxgeom

                    if hasattr(oxgeom, "geometries_from_polygon"):
                        fetched = oxgeom.geometries_from_polygon(nigeria_poly, tags)
                except Exception:
                    pass
            # final fallback: geometries_from_bbox using nigeria_gdf.bounds
            if fetched is None:
                minx, miny, maxx, maxy = nigeria_gdf.total_bounds  # (minx, miny, maxx, maxy)
                # note: geometries_from_bbox signature sometimes (north, south, east, west) in old script; prefer geometries_from_bbox(miny, minx, maxy, maxx) or use named args
                if hasattr(ox, "geometries_from_bbox"):
                    fetched = ox.geometries_from_bbox(maxy, miny, maxx, minx, tags)  # best-effort
                else:
                    # try submodule
                    from osmnx import geometries as oxgeom

                    if hasattr(oxgeom, "geometries_from_bbox"):
                        fetched = oxgeom.geometries_from_bbox(maxy, miny, maxx, minx, tags)
            if fetched is None:
                raise RuntimeError("No suitable geometries_from_* function found in osmnx.")
        except Exception as e:
            raise RuntimeError(f"OSMnx geometry fetch failed: {e}")
        # filter for LineString-like geometries
        if fetched is not None and not fetched.empty:
            fetched = fetched[fetched.geometry.notnull() & fetched.geometry.type.isin(["LineString", "MultiLineString"])]
            combined.append(fetched.copy())
    if combined:
        concatenated = pd.concat(combined, ignore_index=True, sort=False)
        return gpd.GeoDataFrame(concatenated, geometry="geometry", crs="EPSG:4326")
    else:
        return gpd.GeoDataFrame({"geometry": []}, geometry="geometry", crs="EPSG:4326")


def get_pipeline_lines(local_path: Optional[Path] = None) -> gpd.GeoDataFrame:
    """Load pipeline/powerline geometries: prefer explicit local_path, else autodiscover, else OSM fallback."""
    # 0) If user supplied explicit path variable use it
    if local_path:
        p = Path(local_path)
        if p.exists():
            lines = try_read_vector(p)
            if lines is not None:
                log.info("Loaded local lines from explicit path: %s", p)
                return lines
            else:
                log.warning("Failed to read provided local path: %s", p)

    # 1) If constants are set via config prefer them
    for option in (LOCAL_PIPELINES_PATH, LOCAL_TRANSMISSION_PATH):
        if option:
            optp = Path(option)
            if optp.exists():
                lines = try_read_vector(optp)
                if lines is not None:
                    log.info("Loaded local lines from configured path: %s", optp)
                    return lines

    # 2) autodiscover under data/raw/
    candidates = autodiscover_line_files(RAW_DIR)
    if candidates:
        for cand in candidates:
            lines = try_read_vector(cand)
            if lines is not None and len(lines) > 0:
                log.info("Auto-discovered and loaded '%s' (%d features)", cand.name, len(lines))
                return lines
        log.info("Autodiscovery found candidates but none loaded successfully. Candidates: %s", [c.name for c in candidates])
    else:
        log.info("No local candidates found under %s", RAW_DIR)

    # 3) OSM fallback (may fail if osmnx missing or network unavailable)
    try:
        log.info("Attempting to fetch pipeline and power lines from OSM via osmnx (may take time).")
        tags_pipe = {"man_made": "pipeline"}
        tags_power = {"power": "line"}
        combined = _get_osmnx_geometries_for_nigeria(tags_pipe, tags_power)
        if combined is not None and len(combined) > 0:
            log.info("Fetched %d features from OSM for Nigeria.", len(combined))
            return combined
        else:
            log.warning("OSM fetch returned zero features.")
            return gpd.GeoDataFrame({"geometry": []}, geometry="geometry", crs="EPSG:4326")
    except Exception as exc:
        log.warning("Could not fetch lines automatically via OSM: %s", exc)
        return gpd.GeoDataFrame({"geometry": []}, geometry="geometry", crs="EPSG:4326")


# -----------------------------
# Main pipeline
# -----------------------------
def nearest_line_index(point_geom, lines, sindex):
    """Return (index_label, min_distance) for nearest line to point_geom using search index."""
    try:
        possible_idx = list(sindex.intersection(point_geom.bounds))
        if not possible_idx:
            # no candidates, compute full distances (slow)
            dists = lines.geometry.distance(point_geom)
            idxmin = dists.idxmin()
            return idxmin, float(dists.loc[idxmin])
        subset = lines.iloc[possible_idx]
        dists = subset.geometry.distance(point_geom)
        idxmin_local = dists.idxmin()
        # idxmin_local is index label in subset; map back to overall index label
        idx_label = subset.index[idxmin_local] if isinstance(idxmin_local, int) and idxmin_local < len(subset.index) else idxmin_local
        return idx_label, float(dists.loc[idxmin_local])
    except Exception as e:
        log.debug("nearest_line_index failure: %s", e)
        # fallback full search
        dists = lines.geometry.distance(point_geom)
        idxmin = dists.idxmin()
        return idxmin, float(dists.loc[idxmin])


def main():
    summary = {"assets_count": 0, "pipelines_count": 0, "nodes_count": 0, "exposure_computed": False}
    # 1. Load or create assets CSV
    try:
        if not ASSETS_CSV.exists() or not is_csv_valid(ASSETS_CSV):
            if ASSETS_CSV.exists():
                log.warning("CSV at %s appears corrupted. Regenerating from template.", ASSETS_CSV)
                ASSETS_CSV.unlink()
            ASSETS_CSV.write_text(ASSETS_CSV_CONTENT, encoding="utf-8")
            log.info("Wrote template assets CSV to %s", ASSETS_CSV)
        else:
            log.info("Assets CSV valid at %s", ASSETS_CSV)
        assets_gdf = load_assets(ASSETS_CSV)
        # write raw assets first
        assets_gdf.to_file(ASSETS_GPKG, layer="assets", driver="GPKG")
        summary["assets_count"] = len(assets_gdf)
        log.info("Wrote assets GeoPackage to %s (CRS EPSG:4326) — %d rows", ASSETS_GPKG, len(assets_gdf))
    except Exception:
        log.error("Failed while loading or writing assets. See traceback.")
        traceback.print_exc()
        sys.exit(1)

    # 2. Load pipeline/transmission geometries
    try:
        lines_gdf = get_pipeline_lines()  # autodiscover if not provided
        if lines_gdf is None or len(lines_gdf) == 0:
            log.warning("No pipeline/transmission geometries available; continuing with asset points only.")
            lines_gdf = gpd.GeoDataFrame({"geometry": []}, geometry="geometry", crs="EPSG:4326")
        else:
            # ensure geometry column exists and crs set
            if "geometry" not in lines_gdf.columns:
                raise RuntimeError("Loaded lines have no geometry column.")
            if lines_gdf.crs is None:
                lines_gdf.set_crs(epsg=4326, inplace=True)
            # coerce multi-line strings to consistent type
            lines_gdf = lines_gdf.explode(index_parts=False).reset_index(drop=True)
            summary["pipelines_count"] = len(lines_gdf)
            log.info("Loaded %d pipeline/powerline features", len(lines_gdf))
    except Exception:
        log.error("Error obtaining pipeline lines.")
        traceback.print_exc()
        sys.exit(1)

    # 3. Snap assets to nearest line & compute snap distance (metric CRS)
    try:
        if lines_gdf is not None and len(lines_gdf) > 0:
            # project to metric CRS for distance math
            lines_m = lines_gdf.to_crs(epsg=3857)
            assets_m = assets_gdf.to_crs(epsg=3857)

            lines_sindex = lines_m.sindex

            nearest_idxs = []
            snap_dists = []
            snap_geoms_metric = []
            for idx, asset in assets_m.iterrows():
                if asset.geometry is None or asset.geometry.is_empty:
                    nearest_idxs.append(None)
                    snap_dists.append(np.nan)
                    snap_geoms_metric.append(None)
                    continue
                try:
                    li, dist_m = nearest_line_index(asset.geometry, lines_m, lines_sindex)
                    # li may be an index label (not necessarily integer positional); safe .loc
                    try:
                        chosen_line = lines_m.loc[li].geometry
                    except Exception:
                        # fallback using iloc if li is positional
                        chosen_line = lines_m.iloc[int(li)].geometry
                    projected = chosen_line.interpolate(chosen_line.project(asset.geometry))
                    nearest_idxs.append(li)
                    snap_dists.append(dist_m)
                    snap_geoms_metric.append(projected)
                except Exception as e:
                    log.debug("snap for asset %s failed: %s", asset.get("Asset_ID", idx), e)
                    nearest_idxs.append(None)
                    snap_dists.append(np.nan)
                    snap_geoms_metric.append(None)

            assets_m["nearest_line_idx"] = nearest_idxs
            assets_m["snap_dist_m"] = snap_dists
            # set geometry to snapped metric point where available
            new_geoms = []
            for i in range(len(assets_m)):
                if snap_geoms_metric[i] is not None:
                    new_geoms.append(snap_geoms_metric[i])
                else:
                    new_geoms.append(assets_m.iloc[i].geometry)
            assets_m["geometry"] = new_geoms
            assets_out = assets_m.to_crs(epsg=4326)
            assets_out.to_file(ASSETS_GPKG, layer="assets", driver="GPKG")
            log.info("Wrote snapped assets to %s", ASSETS_GPKG)
            # export pipelines for inspection (original CRS)
            try:
                lines_gdf.to_file(PIPELINES_GEOJSON, driver="GeoJSON")
                log.info("Wrote pipelines/transmission geometries to %s", PIPELINES_GEOJSON)
            except Exception as e:
                log.warning("Could not write pipelines GeoJSON: %s", e)
        else:
            assets_gdf.to_file(ASSETS_GPKG, layer="assets", driver="GPKG")
            log.info("Wrote assets (no lines available) to %s", ASSETS_GPKG)
    except Exception:
        log.error("Error during snapping step; see traceback.")
        traceback.print_exc()
        sys.exit(1)

    # 4. Compute pipeline lengths
    try:
        if lines_gdf is not None and len(lines_gdf) > 0:
            lines_out = lines_gdf.to_crs(epsg=3857).copy()
            lines_out["length_m"] = lines_out.geometry.length
            lines_out["length_km"] = lines_out["length_m"] / 1000.0
            lines_out["pipe_id"] = lines_out.index.astype(str)
            try:
                lines_out.to_crs(epsg=4326).to_file(PIPELINES_WITH_LENGTHS, driver="GeoJSON")
                log.info("Wrote pipelines with lengths to %s", PIPELINES_WITH_LENGTHS)
            except Exception as e:
                log.warning("Could not write pipelines with lengths: %s", e)
        else:
            log.info("No pipelines to compute lengths for.")
    except Exception:
        log.error("Error computing pipeline lengths.")
        traceback.print_exc()
        sys.exit(1)

    # 5. Build NetworkX graph & centralities
    try:
        if lines_gdf is not None and len(lines_gdf) > 0:
            lines_m = lines_gdf.to_crs(epsg=3857)
            edges = []
            for idx, row in lines_m.iterrows():
                geom = row.geometry
                if geom is None or geom.is_empty:
                    continue
                parts = list(geom.geoms) if geom.geom_type == "MultiLineString" else [geom]
                for part in parts:
                    coords = list(part.coords)
                    for i in range(len(coords) - 1):
                        a = coords[i]
                        b = coords[i + 1]
                        a_key = (round(a[0], 6), round(a[1], 6))
                        b_key = (round(b[0], 6), round(b[1], 6))
                        dist = LineString([a, b]).length
                        edges.append((a_key, b_key, {"length_m": dist, "feature_type": row.get("feature_type", "line")}))

            G = nx.Graph()
            for u, v, attr in edges:
                G.add_edge(u, v, **attr)
            log.info("Built graph: %d nodes, %d edges", G.number_of_nodes(), G.number_of_edges())

            deg = nx.degree_centrality(G)
            if G.number_of_nodes() > 2000:
                log.info("Large graph detected; computing approximate betweenness (k=200)")
                btw = nx.betweenness_centrality(G, k=200, normalized=True, seed=42)
            else:
                btw = nx.betweenness_centrality(G, normalized=True)

            nodes_list = []
            for key in deg.keys():
                x_m, y_m = key
                nodes_list.append({"x_m": x_m, "y_m": y_m, "deg_centrality": deg[key], "btw_centrality": btw.get(key, 0.0)})
            nodes_df = pd.DataFrame(nodes_list)
            if not nodes_df.empty:
                nodes_gdf = gpd.GeoDataFrame(nodes_df, geometry=gpd.points_from_xy(nodes_df.x_m, nodes_df.y_m), crs="EPSG:3857")
                nodes_geo = nodes_gdf.to_crs(epsg=4326)
                try:
                    nodes_geo.to_file(NODES_GPKG, layer="nodes", driver="GPKG")
                    nodes_geo_csv = nodes_geo.copy()
                    nodes_geo_csv["lon"] = nodes_geo_csv.geometry.x
                    nodes_geo_csv["lat"] = nodes_geo_csv.geometry.y
                    nodes_geo_csv[["lon", "lat", "deg_centrality", "btw_centrality"]].to_csv(NODES_CSV, index=False)
                    log.info("Wrote node centralities to %s", NODES_CSV)
                except Exception as e:
                    log.warning("Could not write node outputs: %s", e)
            summary["nodes_count"] = G.number_of_nodes()
        else:
            log.info("Skipping graph build: no line geometries.")
    except Exception:
        log.error("Error building graph or computing centralities.")
        traceback.print_exc()
        sys.exit(1)

    # 6. Exposure join (spatial join + nearest distance)
    try:
        if not ZENODO_CSV_PATH.exists():
            log.warning("Zenodo CSV not found at %s. Skipping exposure computation.", ZENODO_CSV_PATH)
        else:
            events_gdf = load_zenodo_events(ZENODO_CSV_PATH)
            if events_gdf is None or len(events_gdf) == 0:
                log.warning("No events loaded from Zenodo CSV; skipping exposure.")
            else:
                events_proj = events_gdf.to_crs(epsg=3857)

                # use snapped assets if available
                try:
                    assets_for_exposure = gpd.read_file(ASSETS_GPKG, layer="assets")
                except Exception:
                    assets_for_exposure = assets_gdf
                assets_proj = assets_for_exposure.to_crs(epsg=3857)

                # count events within 10 km using spatial join
                events_buffered = events_proj.copy()
                events_buffered["geometry"] = events_buffered.geometry.buffer(10000)
                # make sure index name compatibility
                join = gpd.sjoin(assets_proj[["Asset_ID", "geometry"]], events_buffered[["geometry"]], how="left", predicate="intersects")
                counts = join.groupby("Asset_ID").size().rename("events_within_10km")
                assets_proj = assets_proj.merge(counts, left_on="Asset_ID", right_index=True, how="left")
                assets_proj["events_within_10km"] = assets_proj["events_within_10km"].fillna(0).astype(int)

                # nearest distance using sindex candidate selection
                events_sindex = events_proj.sindex
                nearest_dist_list = []
                for idx, asset in assets_proj.iterrows():
                    if asset.geometry is None or asset.geometry.is_empty:
                        nearest_dist_list.append(np.nan)
                        continue
                    # candidate approx search within 20km bbox then exact min
                    bbox = asset.geometry.buffer(20000).bounds
                    possible = list(events_sindex.intersection(bbox))
                    if possible:
                        dists = events_proj.iloc[possible].geometry.distance(asset.geometry)
                        nearest_dist_list.append(float(dists.min()))
                    else:
                        dists = events_proj.geometry.distance(asset.geometry)
                        nearest_dist_list.append(float(dists.min()))
                assets_proj["nearest_event_dist_m"] = nearest_dist_list

                # save exposure sample
                out_cols = ["Asset_ID", "Name", "Type", "State_Region", "Capacity", "nearest_event_dist_m", "events_within_10km"]
                available_cols = [c for c in out_cols if c in assets_proj.columns]
                out_df = assets_proj[available_cols].copy()
                out_df.to_csv(EXPOSURE_CSV, index=False)
                log.info("Wrote exposure sample to %s", EXPOSURE_CSV)
                summary["exposure_computed"] = True
    except Exception:
        log.error("Could not compute exposure sample.")
        traceback.print_exc()

    # Write summary (light)
    try:
        SUMMARY_JSON.write_text(json.dumps(summary, indent=2), encoding="utf-8")
        log.info("Wrote processing summary to %s", SUMMARY_JSON)
    except Exception:
        log.warning("Could not write summary JSON.")

    log.info("Script finished. Inspect outputs in %s", PROC_DIR)


if __name__ == "__main__":
    main()

/var/folders/_l/rhclkbys63zdyq4761mjl8xm0000gp/T/ipykernel_82742/1472111764.py:35: FutureWarning: This function has no longer any effect, and will be removed in a future release. Starting with Shapely 2.0, equivalent speedups are always available
  shapely.speedups.enable()
INFO: Assets CSV valid at data/raw/oil_gas_assets.csv
INFO: Cleaned CSV written to: data/raw/oil_gas_assets.csv.cleaned
INFO: Created 19 records
INFO: Wrote assets GeoPackage to data/processed/oil_gas_assets.gpkg (CRS EPSG:4326) — 19 rows
INFO: Attempting to read vector file: data/raw/powerplants.geojson.zip
INFO: Attempting to read vector file: data/raw/Global_Oil_and_Gas_Features_2949233499042865256.zip
INFO: Attempting to read vector file: data/raw/ne_10m_admin_1_states_provinces.zip
INFO: Attempting to read vector file: data/raw/Nigeria Electricity Transmission Network.shp
INFO: Auto-discovered and loaded 'Nigeria Electricity Transmission Network.shp' (116 features)
INFO: Loaded 161 pipeline/powerline features
I